# Code pour la sélection et le traitement d'images pour le calcul d'indices spectraux

> Calculs du NDVI

> Monthly NDVI 2017-2024

In [275]:
##ACCES AU DRIVE pour lecture des fichiers en pré-requis
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [276]:
import ee
import folium
import geemap
import sys
import os

In [277]:
try:
        ee.Initialize(project='ee-ninabegue-tdv')
except Exception as e:
        # Trigger the authentication flow.
        ee.Authenticate()
        # Initialize the library.
        ee.Initialize(project='ee-ninabegue-tdv')

In [278]:
# @title Choisis
do_images = False # @param {"type":"boolean"}

do_NDVI = False # @param {"type":"boolean"}
do_mean_NDVI = False # @param {"type":"boolean"}

do_mean_MNDWI = False # @param {"type":"boolean"}

do_month = True # @param {"type":"boolean"}




In [279]:
# @title Pilote site
do_FR = False # @param {"type":"boolean"}
do_DU = False # @param {"type":"boolean"}
do_DA = False # @param {"type":"boolean"}
do_ES = False # @param {"type":"boolean"}
do_sebou = False # @param {"type":"boolean"}
do_CU = True # @param {"type":"boolean"}

## **MAP INTERFACE**

In [280]:
map = geemap.Map(ee_initialize=False)
#map.add_basemap('Google Satellite')
map.setControlVisibility()
map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…

## 1/ Chargement de la collection d'images

In [281]:
bands=['B2','B3','B4','B8','B8A','B11','B12', 'SCL'];

### Site test

**Tuile par tuile** car si geometry est trop grande, on obtient le message d'erreur "User memory limit exceeded" par la suite

In [282]:
import geopandas as gpd
import re

In [283]:
if do_FR is True:
  # dst_crs = 'EPSG:2154'#Durance
  pilote='Camargue'
  dist_th = 10
if do_CU is True:
  pilote='Curonian_Lagoon'
  dist_th = 50
if do_DU is True:
  pilote='DU'
  dist_th = 5
if do_DA is True:
  # dst_crs = 'EPSG:2100'#Struma
  pilote='Danube_delta'
  dist_th = 10
if do_ES is True:
  # dst_crs = 'EPSG:4326'
  pilote='Valencia'
  dist_th = 5
if do_sebou is True:
  pilote='SEBOU'
  dist_th = 50

In [284]:
dossier = 'R4Cs_WP6_layers'
path_to_data=f"/content/drive/MyDrive/{dossier}/{pilote}/"
# tile="ID_249"
# path_to_shp=path_to_data+tile+".shp"
os.listdir(path_to_data)

['Site_Curonian_lagoon.gpkg',
 'R4Cs_WP6_layers_export_S2_GRANULE_IDs_SiteCuronianlagoongpkg_2017-01-01_2024-12-31.csv',
 'Site_Curonian_lagoon_2017-01-01_2024-12-31_mean_ndvi_mon01.tif',
 'Site_Curonian_lagoon_2017-01-01_2024-12-31_mean_ndvi_mon02.tif',
 'Site_Curonian_lagoon_2017-01-01_2024-12-31_mean_ndvi_mon03.tif',
 'Site_Curonian_lagoon_2017-01-01_2024-12-31_mean_ndvi_mon04.tif',
 'Site_Curonian_lagoon_2017-01-01_2024-12-31_mean_ndvi_mon05.tif',
 'Site_Curonian_lagoon_2017-01-01_2024-12-31_mean_ndvi_mon06.tif',
 'Site_Curonian_lagoon_2017-01-01_2024-12-31_mean_ndvi_mon07.tif',
 'Site_Curonian_lagoon_2017-01-01_2024-12-31_mean_ndvi_mon08.tif',
 'Site_Curonian_lagoon_2017-01-01_2024-12-31_mean_ndvi_mon09.tif',
 'Site_Curonian_lagoon_2017-01-01_2024-12-31_mean_ndvi_mon10.tif',
 'Site_Curonian_lagoon_2017-01-01_2024-12-31_mean_ndvi_mon12.tif',
 'Site_Curonian_lagoon_2017-01-01_2024-12-31_mean_ndvi_mon11.tif']

In [285]:
tiles = [f for f in os.listdir(path_to_data) if f.endswith(".gpkg")]
tiles

['Site_Curonian_lagoon.gpkg']

In [286]:
tiles[0].split('.')[0]

'Site_Curonian_lagoon'

In [287]:
def remove_special_characters(text):
    return re.sub(r'[^a-zA-Z0-9]', '', text)

In [288]:
def polygon_to_ee(polygon):
    return [[[x, y] for x, y, *_ in polygon.exterior.coords]]

In [289]:
# Lire les fichiers et les convertir en FeatureCollection
features_buff = []
features = []
for file in tiles:
    gdf = gpd.read_file(path_to_data + file).to_crs(epsg=4326)
    for _, row in gdf.iterrows():
        geom = row.geometry
        geom = geom.buffer(0.001)
        geom_buff = geom.buffer(0.003)
         # Modification here: Ensure coords is a list of lists of lists for MultiPolygon
        if geom_buff.geom_type == 'MultiPolygon':
            coords_buff = [polygon_to_ee(poly) for poly in geom_buff.geoms]
        else:  # geom_buff is a Polygon
            coords_buff = [polygon_to_ee(geom_buff)]

        # Handle both Polygon and MultiPolygon for geom
        if geom.geom_type == 'MultiPolygon':
            coords = [polygon_to_ee(poly) for poly in geom.geoms]
        else: # geom is a Polygon
            coords = [polygon_to_ee(geom)]
        ee_geom_buff = ee.Geometry.MultiPolygon(coords_buff)
        ee_geom = ee.Geometry.MultiPolygon(coords)
        features_buff.append(ee.Feature(ee_geom_buff).set({"ID" : tiles[0].split('.')[0]}))#.set({"ID" : remove_special_characters(row.nom_bv)}))
        features.append(ee.Feature(ee_geom).set({"ID" : tiles[0].split('.')[0]}))
features_buff

In [290]:
row

,0
ID,1000000000101
SITECODE,LTSIU0012
GKODAS,1
VIETA,"Klaip?dos raj., Šilut?s raj., Nerijos savivald..."
LAT,55.392834
...,...
nat,NATURA 2000
auxiliar_6,314777.312042
auxiliar_7,6138415.697301
auxiliar_8,295.848872


In [291]:
# Créer une FeatureCollection
geometry = ee.FeatureCollection(features_buff)
geometry_to_export = ee.FeatureCollection(features)
# geometry

In [292]:
#add to map
map.centerObject(geometry.first(),4)
map.addLayer(geometry,{},'shp_buff')
map.addLayer(geometry_to_export,{},'shp')

### Sélection

In [293]:
# startDate='2016-01-01';####sentinel commence en mars 2017 sur GEE
startDate = '2017-01-01'
endDate='2024-12-31';

colS2A=ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED").filterDate(startDate,endDate).filterBounds(geometry).select(bands)#.filter(ee.Filter.eq('CLOUDY_PIXEL_PERCENTAGE',0))

# n=len(colS2A.getInfo()["features"])
# print("Nb images : "+str(n))

In [294]:
# colS2A

In [295]:
dst_crs = colS2A.first().select('B2').projection()

In [296]:
dst_crs

In [297]:
# export_image(colS2A.select(['B2','B3','B4']).mean(),'S2A_mean_'+remove_special_characters(tiles[0])+'_'+startDate+'_'+endDate+'_1',dossier,geometry.geometry())

#### Correction nuages

In [298]:
colS2_cloud = ee.ImageCollection("COPERNICUS/S2_CLOUD_PROBABILITY").filterDate(startDate,endDate).filterBounds(geometry)#.filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE',1))

In [299]:
# export_image(colS2_cloud.select(['probability']).mean(),'S2_cloud_'+remove_special_characters(tiles[0])+'_'+startDate+'_'+endDate+'_1',dossier,geometry.geometry())

In [300]:
# colS2_cloud

In [301]:
#/ Associer chaque image S2_SR avec son masque S2_CLOUD_PROBABILITY

join_filter_index = ee.Filter.equals(
    leftField= "system:index",
    rightField= "system:index"
  )

In [302]:
joined = ee.Join.inner().apply(
  primary= colS2A,
  secondary= colS2_cloud,
  condition= join_filter_index
  )

# max_diff = 1000 * 60 * 60  # 1 heure en millisecondes
# join_filter = ee.Filter.maxDifference(
#   difference= max_diff,
#   leftField= 'system:time_start',
#   rightField= 'system:time_start'
#   )

# inner_join = ee.Join.saveBest('match', 'time_diff')
# joined = inner_join.apply(colS2A, colS2_cloud, join_filter)

In [303]:
def match_collections(feature) :
  primary = ee.Image(feature.get("primary"));
  secondary = ee.Image(feature.get("secondary"));
  return primary.addBands((secondary.select("probability")).rename('cloud_probability')).copyProperties(primary, primary.propertyNames())

In [304]:
#// Convertir le résultat en collection
colS2A_cloud = ee.ImageCollection(joined.map(match_collections))

In [305]:
# export_image(colS2A_cloud.select(['B2','B3','B4','cloud_probability']).mean(),'S2A_cloud_'+remove_special_characters(tiles[0])+'_'+startDate+'_'+endDate+'_1',dossier,geometry.geometry())

In [306]:
# colS2A_cloud

In [307]:
#// Fonction pour ajouter la couverture nuageuse à chaque image de colS2A_cloud
def addCloudScore(image):
  cloudScore = image.select("cloud_probability").reduceRegion(
    reducer= ee.Reducer.mean(),
    geometry= image.geometry(),
    scale= 200
  ).get("cloud_probability");
  return image.set("cloud_score", cloudScore).copyProperties(image)

In [308]:
#// Ajouter le score de couverture nuageuse
colS2A_cloud_scored = colS2A_cloud.map(addCloudScore);

In [309]:
# colS2A_cloud_scored

In [310]:
#// Filtrer les images avec moins de 10% de nuages
colS2A_filtered = ee.ImageCollection(colS2A_cloud_scored.filter(ee.Filter.lt("cloud_score", 20)))

In [311]:
# colS2A_filtered

In [312]:
# export_image(colS2A_filtered.select(['B2','B3','B4']).mean(),'S2A_filtered_'+remove_special_characters(tiles[0])+'_'+startDate+'_'+endDate+'_1',dossier,geometry.geometry())

In [313]:
#// Appliquer le masque sur les images restantes
def maskClouds(image):
  cloudMask = image.select('cloud_probability').lt(20)
  return image.updateMask(cloudMask).copyProperties(image)

In [314]:
#// Appliquer le masque aux images Sentinel-2
colS2A_cloud_masked = colS2A_filtered.map(maskClouds);

In [315]:
# colS2A_clean

In [316]:
# n=len(colS2A_clean.getInfo()["features"])
# print("Nb images : "+str(n))

In [317]:
# ── Masque SCL (nuages épais, fins, ombres, neige) ───────────────────────────
# SCL est incluse dans S2_SR_HARMONIZED pour toute la série 2017-2024
# Valeurs SCL à masquer : 1=saturé, 3=ombre nuage, 8=nuage moyen,
#                         9=nuage fort, 10=nuage fin, 11=neige
SCL_INVALID = [1, 3, 8, 9, 10, 11]

def maskSCL(image):
    scl = image.select('SCL')
    # Construire un masque : 1 = pixel valide, 0 = à masquer
    valid = scl.neq(SCL_INVALID[0])
    for val in SCL_INVALID[1:]:
        valid = valid.And(scl.neq(val))
    return image.updateMask(valid).copyProperties(image, image.propertyNames())

In [318]:
colS2A_clean = colS2A_cloud_masked.map(maskSCL)

#### Composition

In [319]:
task_compo = ee.batch.Export.table.toDrive(
    collection=colS2A_clean.map(lambda img: ee.Feature(None, {'GRANULE_ID': img.get('GRANULE_ID')})),
    description=dossier+'_export_S2_GRANULE_IDs_'+remove_special_characters(tiles[0])+'_'+startDate+'_'+endDate,
    folder=dossier,
    fileFormat='CSV'
)

task_compo.start()
print(task_compo.status())

{'state': 'READY', 'description': 'R4Cs_WP6_layers_export_S2_GRANULE_IDs_SiteCuronianlagoongpkg_2017-01-01_2024-12-31', 'priority': 100, 'creation_timestamp_ms': 1779968612986, 'update_timestamp_ms': 1779968612986, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_FEATURES', 'id': '7ZSZ4M7LLL5NOAPEUDBNRDDQ', 'name': 'projects/ee-ninabegue-tdv/operations/7ZSZ4M7LLL5NOAPEUDBNRDDQ'}


In [320]:
# count_composite = ee.Image(colS2A_clean.count()) #image à 10 bandes avec nombre d'images valides par pixel pour chaque bande

In [321]:
map.addLayer(colS2A_clean.mean(),{'min':0, 'max':2000},'S2A_mean')
map

Map(center=[55.38962828483359, 21.163808936062406], controls=(WidgetControl(options=['position', 'transparent_…

In [322]:
# export_image(colS2A_clean.select(['B2','B3','B4']).mean(),'S2A_mean_'+remove_special_characters(tiles[0])+'_'+startDate+'_'+endDate,dossier,geometry.geometry())

### Affichage sur MAP interface

In [323]:
def visualisations_images(collection_img,my_map,geometry,step,band):
  collection = collection_img.limit(10)
  n=len(collection.getInfo()["features"])
  collection=collection.toList(n)

  if len(band)==1:
    print('1 band')
    visArgs = {'bands': band, 'min': 0, 'max': 1}
  else :
    visArgs = {'bands': band, 'min': 0, 'max': 4000}

  for i in range(0,n,step):
    image = ee.Image(collection.get(i))
    name = image.get('GRANULE_ID').getInfo()
    my_map.addLayer(image.clip(geometry), visArgs, name)

In [324]:
# visualisations_images(colS2A_final,map,geometry,1,['B4','B3','B2'])

In [325]:
# visualisations_images(colS2A_cloud,map,geometry,1,['probability'])

### Collection finale

In [326]:
def harmonize_to_uint16(image):
    # Harmoniser les types de bandes (cloud_probability est en Byte -> converti en Uint16)
    cloud = image.select('cloud_probability').toUint16()
    others = image.select(['B2', 'B3', 'B4', 'B8', 'B8A', 'B11', 'B12']).toUint16()

    return others.addBands(cloud).reproject(crs = dst_crs, scale = 10)

Collection harmonisée en terme de format de donnnées et dans un système de projection adéquat :

In [327]:
colS2A_final = colS2A_clean.map(harmonize_to_uint16)
# count_composite = harmonize_to_uint16(count_composite)

In [328]:
# visualisations_images(colS2A_final,map,geometry,1,['B4','B3','B2'])

In [329]:
# colS2A_final

### Mosaiquage pour les dates identiques

In [330]:
# colS2A_clean_to_export = colS2A_clean.map(harmonize_to_uint16)
# colS2A_to_export = colS2A_filtered.map(harmonize_to_uint16)

In [331]:
# # Fonction pour grouper les images par date et créer des mosaïques
# def mosaic_by_date(date):
#     date_str = ee.Date(date).format("YYYY-MM-dd")
#     name = ee.String("Mosaic_").cat(date_str)

#     images_on_date = colS2A_clean.filterDate(date, ee.Date(date).advance(1, "day"))###sur la collection d'images S2 brut
#     mosaic = images_on_date.mosaic().set({
#         "date": date_str,
#         "mosaic_name": name
#     })  # Ajout des métadonnées

#     return mosaic.clip(images_on_date.geometry()).reproject(crs = dst_crs, scale = 10).copyProperties(mosaic)

In [332]:
# # Obtenir les dates uniques dans la collection
# dates = colS2A_final.aggregate_array("system:time_start").distinct()
# # dates

In [333]:
# # Créer une collection de mosaïques par date
# mosaic_collection = ee.ImageCollection(dates.map(mosaic_by_date))
# mosaic_collection

In [334]:
# # Fonction pour créer une mosaïque par date
# def mosaic_by_date_clouds(date):
#     date_str = ee.Date(date).format("YYYY-MM-dd")
#     next_date = ee.Date(date).advance(1, 'day')
#     name = ee.String("Mosaic_").cat(date_str)

#     filtered = colS2A_cloud.filterDate(date, next_date)

#     mosaic = filtered.mosaic().set({
#         'system:time_start': ee.Date(date).millis(),
#         "mosaic_name": name
#     })
#     return mosaic

In [335]:
# # Créer une collection de mosaïques par date
# mosaic_collection_clouds = ee.ImageCollection(dates.map(lambda d: mosaic_by_date_clouds(d)))
# mosaic_collection_clouds

In [336]:
# @title Exportation sur drive : { display-mode: "form" }
do_continue = True # @param {type:"boolean"}
#do_continue=bool(int(input("Continue to classification by RF ? (0 or 1) ")))

if do_continue==False:
  raise SystemExit

## 3/ Calcul d'indices et compilation

### NDVI

In [337]:
def calcul_ndvi(eeim):
  return eeim.normalizedDifference(['B8','B4']).rename('ndvi').copyProperties(eeim, eeim.propertyNames())

In [338]:
colS2A_ndvi = colS2A_final.map(calcul_ndvi)

In [339]:
# colS2A_ndvi

In [340]:
def mndwi_s2(eeim):##GREEN & SWIR1
  SWIR = eeim.select("B11")################SWIR1
  GREEN = eeim.select("B3")
  #MNDWI normalized and thresholded :
  #MNDWI = (1 - (SWIR / Green)) / (1 + (SWIR / Green))
  #MNDWI = (constant - rapport) / (constant + rapport)
  #MNDWI = up / down
  constant = ee.Image(int(1))
  rapport = SWIR.divide(GREEN)
  up = constant.subtract(rapport)
  down = constant.add(rapport)
  mndwi=up.divide(down)
  return mndwi.rename(["mndwi"]).copyProperties(eeim, eeim.propertyNames())

In [341]:
colS2A_mndwi = colS2A_final.map(mndwi_s2)

### Correction bords satellite

In [342]:
def feather_mask(img):
    """
    2. Appliquer un masque spatial basé sur la distance aux bords (feathering)
  Créer un "buffer interne" qui masque les pixels trop proches du bord de l’image.
    """
    # Masque binaire des pixels valides
    # 1. Créer un masque géographique (bord spatial de l’image)
    footprint = ee.Image(1).clip(img.geometry()).mask()

    # Distance aux bords de pixels valides (euclidean distance)
    distance = footprint.Not().fastDistanceTransform(30).sqrt()

    # Masque seulement les pixels à plus de N pixels des bords
    safe_mask = distance.gt(dist_th)  # Ajuster seuil selon besoin
    return img.updateMask(safe_mask).copyProperties(img, img.propertyNames())

In [343]:
colS2A_ndvi_corr = colS2A_ndvi.map(feather_mask)

In [344]:
# colS2A_ndvi_corr

In [345]:
colS2A_mndwi_corr = colS2A_mndwi.map(feather_mask)

In [346]:
# map.addLayer(colS2A_ndvi_corr.mean(),{'min':-1,'max':1},'mean_ndvi_corr')

In [347]:
count_mask = colS2A_ndvi_corr.count().reproject(
    crs= dst_crs,
    scale= 10
)

In [348]:
# # Garde les zones avec >5 observations valides
# count_mask = count_mask.gt(10)
# # count_mask

In [349]:
# map.addLayer(count_mask,{'min':0,'max':1},'count_mask')

In [350]:
vegetation_composite = ee.Image(colS2A_ndvi_corr.select(['ndvi']).mean()).toFloat().reproject(
    crs= dst_crs,
    scale= 10
    ).rename(['mean_ndvi'])

In [351]:
vegetation_composite = vegetation_composite.unmask(0).clip(geometry_to_export)

In [352]:
map.addLayer(vegetation_composite,{'min':-1,'max':1},'vegetation_composite')

In [353]:
# colS2A_ndvi_1 = mosaic_collection_corrected.map(calcul_ndvi)

In [354]:
water_composite = ee.Image(colS2A_mndwi_corr.select(['mndwi']).mean()).toFloat().reproject(
    crs= dst_crs,
    scale= 10
    ).rename(['mean_mndwi'])

In [355]:
water_composite = water_composite.unmask(0).clip(geometry_to_export)

In [356]:
map

Map(center=[55.38962828483359, 21.163808936062406], controls=(WidgetControl(options=['position', 'transparent_…

## 3/ Export

In [357]:
# @title Exportation sur drive : { display-mode: "form" }
do_export = True # @param {type:"boolean"}
#do_continue=bool(int(input("Continue to classification by RF ? (0 or 1) ")))

if do_export==False:
  raise SystemExit

In [358]:
def export_image(image,name,dossier,roi):
  #Export the image, specifying the CRS, transform, and region.
  #proj=image.projection().getInfo()
  task=ee.batch.Export.image.toDrive(**{
    'image': image,#.clip(roi),
    'description': name,
    #'crs': proj['crs'],
    'scale': 10,
    'folder': dossier,
    'fileFormat': 'GeoTIFF',
    #'crsTransform': proj['transform'],
    'region': roi,
    'maxPixels': 1e13
    #'formatOptions': {'cloudOptimized': True}
    })
  task.start()
  print('Polling for task (id: {}).'.format(task.id))

In [359]:
def export_images(image,name,dossier):
  #Export the image, specifying the CRS, transform, and region.
  #proj=image.projection().getInfo()
  task=ee.batch.Export.image.toDrive(**{
    'image': image,#.clip(roi),
    'description': name,
    #'crs': proj['crs'],
    'scale': 10,
    'folder': dossier,
    'fileFormat': 'GeoTIFF',
    #'crsTransform': proj['transform'],
    'region': image.geometry(),
    'maxPixels': 1e13
    #'formatOptions': {'cloudOptimized': True}
    })
  task.start()
  print('Polling for task (id: {}).'.format(task.id))

In [360]:
def export_col_img(col_img,suffix,dossier):
  n=len(col_img.getInfo()["features"])
  col_img=col_img.toList(n)
  for i in range(0,n):
    image = ee.Image(col_img.get(i))
    name = image.get('GRANULE_ID').getInfo()+suffix
    print(name)
    task=export_images(image,name,dossier)

In [361]:
tiles

['Site_Curonian_lagoon.gpkg']

In [362]:
# Obtenir la liste des entités de la FeatureCollection
features_list = geometry_to_export.toList(geometry_to_export.size())

In [363]:
dossier

'R4Cs_WP6_layers'

In [364]:
# Boucler sur chaque entité et exporter une image individuelle
for i in range(geometry_to_export.size().getInfo()):
    feature = ee.Feature(features_list.get(i))  # Récupérer la feature
    geom = feature.geometry()
    tile = str(feature.get("ID").getInfo())  # Utilisation d'un identifiant unique
    print(tile)

    folder_name = dossier

    #Mean NDVI : 1 image
    if do_mean_NDVI is True:
      task_mean_ndvi = export_image(vegetation_composite,tile+'_'+startDate+'_'+endDate+'_mean_ndvi',folder_name,geom)
      # task_count = export_image(count_mask,tile+'_'+startDate+'_'+endDate+'_ndvi_count_mask',folder_name,geom)

    #Mean NDVI : 1 image
    if do_mean_MNDWI is True:
      task_mean_mndwii = export_image(water_composite,tile+'_'+startDate+'_'+endDate+'_mean_mndwi',folder_name,geom)
      # task_count = export_image(count_mask,tile+'_'+startDate+'_'+endDate+'_ndvi_count_mask',folder_name,geom)

    #All images
    if do_images is True:
      task_img=export_col_img(colS2A_final,tile+'_'+startDate+'_'+endDate,folder_name)
    #All S2 image collection
    if do_images is True:
      task_img=export_col_img(colS2A_final,'',folder_name)
      #export_col_img(mosaic_collection_clouds,'_clouds',folder_name)

    #All NDVI index
    if do_NDVI is True:
      task_ndvi=export_col_img(colS2A_ndvi,'_ndvi',folder_name)


    #code.earthengine.google.com/tasks pour regarder les exportations en cours

Site_Curonian_lagoon


In [365]:
# Collection mensuelle: 12 images (moyenne NDVI multi-annuelle par mois)
months = ee.List.sequence(1, 12)

# def _monthly_mean(m):
#   m = ee.Number(m)
#   ic = colS2A_ndvi_corr.filter(ee.Filter.calendarRange(m, m, 'month')).select('ndvi')
#   # Si ic est vide => renvoyer une image NDVI entièrement masquée (même bande)
#   mean_img = ic.mean()
#   safe_img = ee.Image(ee.Algorithms.If(
#       ic.size().gt(0),
#       mean_img,
#       ee.Image.constant(0).rename('ndvi').updateMask(ee.Image(0))
#   ))
#   # Harmoniser le type/bande et poser le mois en propriété
#   return (safe_img
#             .toFloat()
#             .rename('mean_ndvi')
#             .set({'month': m}))

In [366]:
# ── Composite annuel de référence (fallback pixels manquants) ────────────────
annual_composite = (colS2A_ndvi_corr
                    .select('ndvi')
                    .mean()
                    .toFloat()
                    .rename('mean_ndvi'))

def _monthly_mean(m):
    m = ee.Number(m)
    ic = colS2A_ndvi_corr.filter(
             ee.Filter.calendarRange(m, m, 'month')).select('ndvi')

    mean_img = ic.mean()

    safe_img = ee.Image(ee.Algorithms.If(
        ic.size().gt(0),
        mean_img,
        ee.Image.constant(0).rename('ndvi').updateMask(ee.Image(0))
    ))

    # ── NOUVEAU : combler les pixels manquants avec le composite annuel ───
    filled_img = safe_img.toFloat().rename('mean_ndvi').unmask(annual_composite)
    # ─────────────────────────────────────────────────────────────────────

    return filled_img.set({'month': m})

monthly_means = ee.ImageCollection.fromImages(months.map(_monthly_mean))

In [367]:
if do_month is True:
  for m in range(1, 13):
    # Récupérer l’image mensuelle (déjà calculée côté serveur)
    monthly_img = ee.Image(
        monthly_means.filter(ee.Filter.eq('month', m)).first()
    )

    # Reprojection + clip sur la géométrie de la tuile
    monthly_img = (monthly_img
                    .reproject(crs=dst_crs, scale=10)
                    .clip(geom))

    out_name = f"{tile}_vegetation_{m:02d}"
    task_month = export_image(monthly_img, out_name, folder_name, geom)

Polling for task (id: DIT6HHKX5B5FZ6TUMB4TRXCI).
Polling for task (id: 36AMJLQUA6DGLKYTTICLLW57).
Polling for task (id: WY3F43AXBQKZVER45GH33XBW).
Polling for task (id: S5ST4FSA3R62N7TBM76KH6M5).
Polling for task (id: IYJS2X45EHWUBZNPYDCYBZ4W).
Polling for task (id: N5BD3G64WGCEPBNAVG5DBJFT).
Polling for task (id: PQKLX2IVT7Q7UATPDDUOQT5R).
Polling for task (id: FCDE3423DH4LLBMFWB2ONCE6).
Polling for task (id: NFVUYYWV7GDKNHE3VR37DJRF).
Polling for task (id: XP6JC5LEUTCXGQT6G47W4AUY).
Polling for task (id: V4EMA4G7Z2QSTMDYDMHBTF35).
Polling for task (id: 5EG6MVU6GN3756ROD327ZO4Q).
